# Deep learning on text

## Load modules from repo

In [ ]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [ ]:
os.getcwd()

In [ ]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_text import preprocess_features
from src.preprocessing.pipelines.deep_learning import load_preprocessors
from src.models.on_text.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

In [ ]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_text)
importlib.reload(src.models.on_text.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

## Load tensorflow

In [ ]:
import tensorflow as tf

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
print(f"tensorflow: {tf.__version__}")

## Load split dataset

In [ ]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')
full_X_train=X_train
full_y_train=y_train

In [ ]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

## Configuration 1

In [ ]:
modality = 'text'
version = 1
artifacts_folder = Path(f'artifacts/on_text/deep_learning/v{version}')
image_artifacts_folder = Path(f'artifacts/on_images/deep_learning/v1')
log_file_path = image_artifacts_folder / 'experiments.parquet'
preprocessors_folder = image_artifacts_folder
tensor_board_folder = image_artifacts_folder / "tensorboard_logs"

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = False  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = True  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

BATCH_SIZE = 32

RANDOM_SEED = 42

# load_model=True
load_model=False


## Preprocessing

In [ ]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [ ]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

In [ ]:
y_train.value_counts(normalize=True).describe()

In [ ]:
y_train.value_counts().describe()

In [ ]:
print(X_train.shape)

In [ ]:
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target'],artifacts_folder=preprocessors_folder)

In [ ]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, full_X_train=full_X_train, full_y_train=full_y_train, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights)
preprocessors |= new_preprocessors

In [ ]:
new_preprocessors

In [ ]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [ ]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False)

## Model

In [ ]:
from tensorflow import keras

### Load or create model

In [ ]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    arch_version = last_experiment.get('arch_version', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    arch_version = last_experiment.get('arch_version', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model = define_model(text_vectorizer=preprocessors['text_vectorizer'], num_classes=27)

In [ ]:
if not load_model:
    arch_version = int(input(f"arch_version? (last: {arch_version})"))

In [ ]:
arch_version

### Summary

In [ ]:
model.summary()

## Callbacks

### ModelCheckpoint

In [ ]:
# # Pick an available filename to save a model.
# arch_version=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{arch_version}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     arch_version+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{arch_version}.h5')
# new_location_for_saving_model


In [ ]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [ ]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [ ]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_accuracy', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='max',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [ ]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [ ]:
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder / f"{timestamp}-{modality}",
    histogram_freq=1  # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [ ]:
import math
typical_minutes_per_epoch=1.2 * X_train.shape[0] / 67932

In [ ]:
# max_epochs=5

# # Calculate expected duration
# available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
# available_minutes

In [ ]:
# Pick max_epochs based on your available time
available_minutes=30

max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

### compilation and callbacks

In [ ]:
learning_rate=0.001

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [ ]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

In [ ]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=total_epochs_trained, callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

## Evaluation

In [ ]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

In [ ]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

In [ ]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [ ]:
#Takes 1m40
y_pred = model.predict(test_ds)

In [ ]:
from sklearn import metrics

In [ ]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [ ]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

In [ ]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

In [ ]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [ ]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

In [ ]:
# Summary over classes
report.iloc[:-3].describe()

In [ ]:
# positive correlation between support and another measure can suggest class imbalance hurts performance
report.corr()

In [ ]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

In [ ]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

## Update tracker

In [ ]:
tracker['comment']="Performance better than the benchmark."
tracker['comment']

In [ ]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Sauvegarde du modèle.")
    keep_candidate=True

    best_epoch_in_session_idx = np.argmax(model_history.history['val_accuracy'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_accuracy = model_history.history['val_accuracy'][best_epoch_in_session_idx]
    tracker['val_accuracy'] = best_val_accuracy

    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_arch-{arch_version}_epoch_index-{best_epoch_global:02d}_val_accuracy-{best_val_accuracy:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_path)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print(f"Effacement de l'ancien modèle chargé {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le modèle chargé.")
    keep_candidate=False


In [ ]:
tracker['epoch_index'] = best_epoch_global
tracker['total_epochs'] = best_epoch_global + 1

In [ ]:
to_track=['arch_version','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate','timestamp','modality']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model, base_model=None)

In [ ]:
tracker

In [ ]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [ ]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [ ]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

In [ ]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, loaded_model=load_model, log_file_path=log_file_path)

## Show tracking logs

In [ ]:
pd.set_option('max_colwidth', None)

In [ ]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0